# Temperature Scaling — Calibración de confianza del modelo

**Objetivo:** encontrar la temperatura T óptima que hace que las
probabilidades del modelo sean más confiables (bien calibradas).

**T < 1.0** → distribución más concentrada → más confianza en la clase ganadora
**T > 1.0** → distribución más plana → menos confianza

Se calibra sobre el **val set** y se evalúa en **test set**.

**Tiempo:** ~10 min. **Runtime:** GPU o CPU (rápido).


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import sys, os, shutil
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
import numpy as np
from scipy.optimize import minimize_scalar

!pip install -q torchxrayvision h5py scipy
print('Setup OK')

CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


## 2. Copiar HDF5 y cargar modelo

In [ ]:
import h5py, shutil, time

H5_LOCAL = '/content/nih_images.h5'
H5_DRIVE = f'{NIH_DIR}/nih_images.h5'

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000):
    print('Copiando HDF5...')
    shutil.copy2(H5_DRIVE, H5_LOCAL)
with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5: {len(hf["images"]):,} imgs')


In [ ]:
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.models.cnn_vit import build_cnn_vit
from src.training.utils import load_config, get_device, load_checkpoint

cfg    = load_config(f'{CODE_DIR}/configs/sprint4.yaml')
device = get_device()

model = build_cnn_vit(cfg).to(device)
ckpt  = load_checkpoint(
    f'{PROJECT_ROOT}/experiments_s4/checkpoints/sprint4_phase2_best.pt',
    map_location=device
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✓ Modelo cargado (epoch {ckpt["epoch"]}, val AUC={ckpt["best_val_auc_macro"]:.4f})')


## 3. Obtener logits en val y test

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_eval_transform

PROCESSED = '/content/drive/MyDrive/Tesis_CXR/data/processed_s4'
eval_tfm  = build_eval_transform(224)

# Copiar CSVs localmente
import pandas as pd
os.makedirs('/content/processed_s4', exist_ok=True)
for csv in ['val.csv', 'test.csv']:
    local = f'/content/processed_s4/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROCESSED}/{csv}', local)
    print(f'✓ {csv}: {len(pd.read_csv(local))} filas')

def get_logits_labels(csv_path):
    """Obtiene logits crudos (sin softmax) y labels del dataset."""
    ds     = NIHDatasetHDF5(csv_path, H5_LOCAL, eval_tfm)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['image'].to(device))
            all_logits.append(logits.cpu())
            all_labels.append(batch['label'])
    return torch.cat(all_logits), torch.cat(all_labels)

print('Obteniendo logits en val...')
val_logits,  val_labels  = get_logits_labels('/content/processed_s4/val.csv')
print(f'  val:  {len(val_labels)} muestras')

print('Obteniendo logits en test...')
test_logits, test_labels = get_logits_labels('/content/processed_s4/test.csv')
print(f'  test: {len(test_labels)} muestras')


## 4. Calibrar temperatura en val

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

def nll_with_temperature(T: float,
                          logits: torch.Tensor,
                          labels: torch.Tensor) -> float:
    """
    Negative Log Likelihood con temperatura T.
    Minimizar esto encuentra la T óptima.
    T < 1.0 → más concentrado (más confianza)
    T > 1.0 → más plano (menos confianza)
    """
    scaled = F.log_softmax(logits / T, dim=1)
    nll    = F.nll_loss(scaled, labels)
    return nll.item()

def accuracy_at_T(T: float, logits, labels) -> float:
    probs = torch.softmax(logits / T, dim=1)
    preds = probs.argmax(dim=1)
    return (preds == labels).float().mean().item()

def mean_confidence_at_T(T: float, logits) -> float:
    probs = torch.softmax(logits / T, dim=1)
    return probs.max(dim=1).values.mean().item()

# Buscar T óptima que minimiza NLL en val
result = minimize_scalar(
    nll_with_temperature,
    bounds=(0.1, 5.0),
    method='bounded',
    args=(val_logits, val_labels),
)
T_optimal = float(result.x)

# Comparar T=1.0 vs T óptima
T_values = np.arange(0.1, 3.1, 0.1)
nlls   = [nll_with_temperature(T, val_logits, val_labels) for T in T_values]
confs  = [mean_confidence_at_T(T, val_logits) for T in T_values]
accs   = [accuracy_at_T(T, val_logits, val_labels) for T in T_values]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(T_values, nlls, 'b-', lw=2)
axes[0].axvline(T_optimal, color='red', linestyle='--',
                label=f'T óptima = {T_optimal:.3f}')
axes[0].axvline(1.0, color='gray', linestyle=':', label='T = 1.0 (actual)')
axes[0].set_xlabel('Temperatura T')
axes[0].set_ylabel('NLL (menor = mejor)')
axes[0].set_title('NLL vs Temperatura')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(T_values, [c*100 for c in confs], 'g-', lw=2)
axes[1].axvline(T_optimal, color='red', linestyle='--')
axes[1].axvline(1.0, color='gray', linestyle=':')
axes[1].set_xlabel('Temperatura T')
axes[1].set_ylabel('Confianza media (%)')
axes[1].set_title('Confianza media vs Temperatura')
axes[1].grid(alpha=0.3)

axes[2].plot(T_values, [a*100 for a in accs], 'r-', lw=2)
axes[2].axvline(T_optimal, color='red', linestyle='--')
axes[2].axvline(1.0, color='gray', linestyle=':')
axes[2].set_xlabel('Temperatura T')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Accuracy vs Temperatura')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Temperature Scaling — T óptima = {T_optimal:.3f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s4/figures/temperature_scaling.png',
            dpi=120, bbox_inches='tight')
plt.show()

print(f'\nRESULTADOS EN VAL:')
print(f'  T = 1.0 (sin scaling):')
print(f'    NLL:       {nll_with_temperature(1.0, val_logits, val_labels):.4f}')
print(f'    Confianza: {mean_confidence_at_T(1.0, val_logits)*100:.1f}%')
print(f'    Accuracy:  {accuracy_at_T(1.0, val_logits, val_labels)*100:.1f}%')
print(f'\n  T = {T_optimal:.3f} (óptima):')
print(f'    NLL:       {nll_with_temperature(T_optimal, val_logits, val_labels):.4f}')
print(f'    Confianza: {mean_confidence_at_T(T_optimal, val_logits)*100:.1f}%')
print(f'    Accuracy:  {accuracy_at_T(T_optimal, val_logits, val_labels)*100:.1f}%')


## 5. Evaluar en test con T óptima

In [ ]:
from sklearn.metrics import roc_auc_score

def compute_auc_macro(logits, labels, T=1.0):
    probs = torch.softmax(logits / T, dim=1).numpy()
    y     = labels.numpy()
    aucs  = []
    for c in range(4):
        y_bin = (y == c).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            aucs.append(roc_auc_score(y_bin, probs[:, c]))
    return float(np.mean(aucs))

print('COMPARATIVA EN TEST:')
print('='*55)

# Sin temperature scaling
auc_1   = compute_auc_macro(test_logits, test_labels, T=1.0)
conf_1  = mean_confidence_at_T(1.0, test_logits)
acc_1   = accuracy_at_T(1.0, test_logits, test_labels)

# Con temperature scaling óptima
auc_T   = compute_auc_macro(test_logits, test_labels, T=T_optimal)
conf_T  = mean_confidence_at_T(T_optimal, test_logits)
acc_T   = accuracy_at_T(T_optimal, test_logits, test_labels)

print(f'{"Métrica":<20} {"T=1.0":>10} {f"T={T_optimal:.2f}":>10} {"Delta":>10}')
print('-'*55)
print(f'{"AUC macro":<20} {auc_1:>10.4f} {auc_T:>10.4f} {auc_T-auc_1:>+10.4f}')
print(f'{"Confianza media":<20} {conf_1*100:>9.1f}% {conf_T*100:>9.1f}% {(conf_T-conf_1)*100:>+9.1f}pp')
print(f'{"Accuracy":<20} {acc_1*100:>9.1f}% {acc_T*100:>9.1f}% {(acc_T-acc_1)*100:>+9.1f}pp')
print()
print('Nota: AUC no cambia con temperature scaling')
print('(AUC es independiente de la escala de probabilidades)')

# Ejemplo: cómo cambian las probabilidades
print()
print('EJEMPLO — cómo cambian las probabilidades:')
idx = 0  # primera muestra de test
p1 = torch.softmax(test_logits[idx:idx+1] / 1.0, dim=1)[0]
pT = torch.softmax(test_logits[idx:idx+1] / T_optimal, dim=1)[0]
print(f'{"Clase":<15} {"T=1.0":>8} {f"T={T_optimal:.2f}":>8}')
for i, name in CLASS_NAMES.items():
    print(f'{name:<15} {p1[i]*100:>7.1f}% {pT[i]*100:>7.1f}%')


## 6. Guardar temperatura y actualizar backend

In [ ]:
import json

# Guardar temperatura en Drive
temp_config = {
    "temperature": round(T_optimal, 4),
    "calibrated_on": "val",
    "val_nll_before": round(nll_with_temperature(1.0, val_logits, val_labels), 4),
    "val_nll_after":  round(nll_with_temperature(T_optimal, val_logits, val_labels), 4),
    "mean_confidence_before": round(float(mean_confidence_at_T(1.0, val_logits)), 4),
    "mean_confidence_after":  round(float(mean_confidence_at_T(T_optimal, val_logits)), 4),
}

# Guardar en Drive
temp_path = f'{PROJECT_ROOT}/experiments_s4/temperature_config.json'
with open(temp_path, 'w') as f:
    json.dump(temp_config, f, indent=2)
print(f'✓ Guardado en Drive: {temp_path}')

# Mostrar el JSON para copiar al backend
print()
print('COPIA ESTE JSON a cxr-system/backend/artifacts/temperature_config.json:')
print()
print(json.dumps(temp_config, indent=2))
print()
print(f'Temperatura óptima: T = {T_optimal:.4f}')
if T_optimal < 1.0:
    print('T < 1.0 → el modelo estaba siendo poco confiante → scaling lo corrige')
else:
    print('T > 1.0 → el modelo estaba siendo overconfident → scaling lo suaviza')


## 7. Cómo aplicar en el backend

Una vez que tengas la temperatura óptima, agrega esto al backend:

```python
# En services/model_service.py

def run_inference(model, tensor, temperature=1.0):
    with torch.no_grad():
        logits = model(tensor)
        # Aplicar temperature scaling antes del softmax
        probs  = torch.softmax(logits / temperature, dim=1)[0].tolist()
    return {
        "probs": probs,
        "predicted_label": int(torch.argmax(torch.tensor(probs)).item()),
    }
```

```python
# En main.py (lifespan), cargar la temperatura:
import json
temp_cfg = json.load(open("artifacts/temperature_config.json"))
app.state.temperature = temp_cfg["temperature"]

# En routers/predict.py, pasar la temperatura:
result = model_service.run_inference(
    model=request.app.state.model,
    tensor=tensor,
    temperature=request.app.state.temperature,
)
```
